# PyTorch Toy Binary Classifier on GPU

This notebook demonstrates how to build, train, and evaluate a deep learning neural network using **PyTorch**, accelerated by a **GPU (CUDA)**.

### Notebook Overview
* **Goal:** Classify non-linearly separable circular data points.
* **Hardware Acceleration:** Leverage CUDA for fast tensor computations and model optimization.
* **Key Components:** Custom PyTorch Neural Network (`nn.Module`), Binary Cross-Entropy Loss, and the Adam Optimizer.

In [ ]:
#!pip install scikit-learn matplotlib

## 1. Setup and Dataset Generation

In this section, we:
* Detect and initialize our GPU device (`cuda`).
* Generate a synthetic circular dataset using `scikit-learn` to simulate a complex, non-linear classification boundary.
* Prepare, partition, and move our data tensors directly into GPU Memory (VRAM).

## 2. Model Architecture and Optimization Setup

Here we construct a Feedforward Neural Network containing:
* An **Input Layer** accepting 2 coordinates.
* **Two Hidden Layers** containing 32 and 16 neurons with **ReLU** activation functions.
* A **Single Output Neuron** utilizing a **Sigmoid** activation function to output binary class probabilities.
* **BCELoss** (Binary Cross Entropy) to evaluate prediction errors and the **Adam** optimizer for weight adjustments.

In [ ]:
import matplotlib.pyplot as plt
import sklearn.datasets
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Setup GPU Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0)})\n")

# 2. Generate Toy Data (Classification Dataset)
# Creates a circular target pattern that a straight linear line cannot solve
X_raw, y_raw = sklearn.datasets.make_circles(
    n_samples=3000, noise=0.05, factor=0.5, random_state=42
)

# Convert arrays into PyTorch Float Tensors
X = torch.tensor(X_raw, dtype=torch.float32)
y = torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1)

# Split into Training (80%) and Test (20%) samples
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# CRUCIAL STEP FOR GPU: Push dataset matrices into VRAM
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)


# 3. Define the Deep Learning Neural Network
class ToyClassifier(nn.Module):

    def __init__(self):
        super(ToyClassifier, self).__init__()
        # Input Layer (2 features) -> Hidden Layer 1 (32 neurons)
        self.layer1 = nn.Linear(2, 32)
        # Hidden Layer 1 -> Hidden Layer 2 (16 neurons)
        self.layer2 = nn.Linear(32, 16)
        # Hidden Layer 2 -> Output Layer (1 probability prediction)
        self.layer3 = nn.Linear(16, 1)
        # Activation Function
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x


# Instantiate the model and map its weights directly onto the GPU
model = ToyClassifier().to(device)

# 4. Set Up Loss Function and Optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 5. Training Loop
epochs = 300
print("--- Starting Training Loop on GPU ---")

for epoch in range(1, epochs + 1):
    model.train()

    # Forward Pass: Compute predictions on the GPU
    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    # Backward Pass: Calculate gradients and adjust network weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print validation progress every 30 epochs
    if epoch % 30 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test)
            # Calculate accuracy percentage
            predicted_classes = (test_preds > 0.5).float()
            accuracy = (predicted_classes == y_test).sum().item() / len(
                y_test
            ) * 100

        print(
            f"Epoch {epoch:03d}/{epochs} | Training Loss: {loss.item():.4f} | Test Accuracy: {accuracy:.2f}%"
        )

print("\nTraining completed successfully!")


Using device: cuda (NVIDIA GeForce GTX 1050 Ti)

--- Starting Training Loop on GPU ---
Epoch 001/300 | Training Loss: 0.6946 | Test Accuracy: 49.17%
Epoch 030/300 | Training Loss: 0.1624 | Test Accuracy: 100.00%
Epoch 060/300 | Training Loss: 0.0061 | Test Accuracy: 100.00%
Epoch 090/300 | Training Loss: 0.0023 | Test Accuracy: 100.00%
Epoch 120/300 | Training Loss: 0.0015 | Test Accuracy: 100.00%
Epoch 150/300 | Training Loss: 0.0011 | Test Accuracy: 100.00%
Epoch 180/300 | Training Loss: 0.0008 | Test Accuracy: 100.00%
Epoch 210/300 | Training Loss: 0.0007 | Test Accuracy: 100.00%
Epoch 240/300 | Training Loss: 0.0005 | Test Accuracy: 100.00%
Epoch 270/300 | Training Loss: 0.0004 | Test Accuracy: 100.00%
Epoch 300/300 | Training Loss: 0.0004 | Test Accuracy: 100.00%

Training completed successfully!


In [ ]:
print(device)

cuda
